# Customer Health Score: Analysing Churn Risk
## Three Approaches to Predicting Churn from Adoption & Usage Data

> **Course:** Enterprise Architecture for SaaS Business Models  
> **Run in:** Google Colab - execute cells top to bottom.

---

## The Problem

You have a dataset of 500 SaaS customers with signals measuring how they use the product - seat utilisation, login frequency, features adopted, QBR recency, support tickets, NPS, and training completion. Some of those customers churned; others renewed or expanded.

**Your goal:** use these signals to predict which customers are at high risk of churning.

There are three distinct approaches. This notebook implements all three. Each section adds new columns to the same dataframe and saves an output file.

| Approach | What it produces | Data needed |
|---|---|---|
| **A - Weighted Score** | Health Score 0–100 per customer | Current snapshot |
| **B - Red Flag Counter** | Flag count + risk tier per customer | Current snapshot |
| **C - Signal Ranking** | Which signals best predict churn | Requires Outcome labels |

---
## Setup - Load the Dataset

Run this cell first. All three option sections depend on it.

In [ ]:
import pandas as pd, numpy as np

!wget -O HealthScore_500.xlsx 'https://github.com/btx4452/CS_HealthScore_Analysis/raw/main/HealthScore_500.xlsx'

df = pd.read_excel('./HealthScore_500.xlsx', sheet_name='Customer Data')
print(f"Loaded {len(df)} rows")
print("Columns:", list(df.columns))

display(df)

---
## Option A - Weighted Health Score (0–100)

**How it works:** each signal is normalised to a 0–100 scale and multiplied by an assigned weight. The weighted values are summed to produce a single Health Score per customer.

| Signal | Column | Weight | Direction |
|---|---|---|---|
| Seat Utilisation Rate | `Seats_Active_30d / Seats_Licensed` | 25% | Higher = better |
| Login Frequency per Seat | `Logins_Total_30d / Seats_Active_30d` | 20% | Higher = better |
| Features Adopted | `Features_Adopted` | 20% | Higher = better |
| QBR Recency | `Last_QBR_Days_Ago` (inverted) | 15% | Lower = better |
| Open Support Tickets | `Support_Tickets_Open` (inverted) | 10% | Lower = better |
| NPS Score | `NPS_Score` | 5% | Higher = better |
| Training Completion | `Training_Completion_pct` | 5% | Higher = better |

**Score thresholds:** `< 40` = High Risk &nbsp;·&nbsp; `40–70` = Monitor &nbsp;·&nbsp; `> 70` = Healthy

**Added columns:** `sig_*` (normalised signals 0–100), `Health_Score`, `Risk_Tier_A`  
**Output file:** `HealthScore_OptionA.xlsx`

In [ ]:
#@title Code Option A - Weighted Health Score
from google.colab import files
pd.options.display.float_format = '{:.2f}'.format

# Normalise each signal to 0–100
df['sig_seat_util'] = ((df['Seats_Active_30d'] / df['Seats_Licensed'].replace(0, 1)).clip(0, 1) * 100).round(1)
df['sig_logins']    = ((df['Logins_Total_30d']  / df['Seats_Active_30d'].replace(0, 1)).clip(0, 30) / 30 * 100).round(1)
df['sig_features']  = ((df['Features_Adopted']  / 5).clip(0, 1) * 100).round(1)
df['sig_qbr']       = ((1 - (df['Last_QBR_Days_Ago'] / 180).clip(0, 1)) * 100).round(1)
df['sig_tickets']   = ((1 - (df['Support_Tickets_Open'] / 10).clip(0, 1)) * 100).round(1)
df['sig_nps']       = (((df['NPS_Score'] + 100) / 200) * 100).round(1)
df['sig_training']  = df['Training_Completion_pct'].clip(0, 100).round(1)

WEIGHTS = {
    'sig_seat_util': 0.25, 'sig_logins':   0.20, 'sig_features': 0.20,
    'sig_qbr':       0.15, 'sig_tickets':  0.10, 'sig_nps':      0.05,
    'sig_training':  0.05,
}
df['Health_Score'] = sum(df[col] * w for col, w in WEIGHTS.items()).round(1)
df['Risk_Tier_A']  = pd.cut(df['Health_Score'], bins=[0, 40, 70, 100],
                             labels=['High Risk', 'Monitor', 'Healthy'],
                             include_lowest=True)

# Summary
print("=== Score distribution by risk tier ===")
print(df.groupby('Risk_Tier_A', observed=True)['Health_Score']
        .agg(['count', 'mean', 'min', 'max']).round(1))

# First 20 customers: key fields → calculated fields → raw fields
KEY  = ['Customer_ID', 'Company_Name', 'Industry', 'ACV_USD']
CALC = ['sig_seat_util', 'sig_logins', 'sig_features', 'sig_qbr',
        'sig_tickets', 'sig_nps', 'sig_training', 'Health_Score', 'Risk_Tier_A']
RAW  = ['Seats_Licensed', 'Seats_Active_30d', 'Logins_Total_30d',
        'Features_Adopted', 'Last_QBR_Days_Ago', 'Support_Tickets_Open',
        'NPS_Score', 'Payment_Overdue_Days', 'Training_Completion_pct']
print("\n=== All customers with calculated columns ===")
display(df[KEY + CALC + RAW].reset_index(drop=True))


---
## Option B - Red Flag Counter

**How it works:** each signal is checked against a binary threshold. If the threshold is breached, one red flag is raised. The total count determines the risk tier.

| Signal | Column | Red Flag Condition |
|---|---|---|
| Seat Utilisation | `Seats_Active_30d / Seats_Licensed` | < 30% |
| Logins per Seat | `Logins_Total_30d / Seats_Active_30d` | < 5 per month |
| Features Adopted | `Features_Adopted` | ≤ 2 |
| Last QBR | `Last_QBR_Days_Ago` | > 90 days |
| Open Support Tickets | `Support_Tickets_Open` | ≥ 3 |
| NPS | `NPS_Score` | < 0 (Detractor) |
| Payment Overdue | `Payment_Overdue_Days` | > 0 |
| Training Completion | `Training_Completion_pct` | < 40% |

**Risk tiers:** `0–1` = Healthy &nbsp;·&nbsp; `2–3` = Watch &nbsp;·&nbsp; `4+` = At Risk

**Added columns:** `flag_*` (1 = flag raised, 0 = clear), `Red_Flag_Count`, `Risk_Tier_B`  


In [ ]:
#@title Option B - Red Flag Counter
from google.colab import files
pd.options.display.float_format = '{:.1f}'.format

seat_util       = (df['Seats_Active_30d'] / df['Seats_Licensed'].replace(0, 1)).round(1)
logins_per_seat = (df['Logins_Total_30d']  / df['Seats_Active_30d'].replace(0, 1)).round(1)

df['flag_seat_util']  = (seat_util < 0.30).astype(int)
df['flag_logins']     = (logins_per_seat < 5).astype(int)
df['flag_features']   = (df['Features_Adopted'] <= 2).astype(int)
df['flag_qbr']        = (df['Last_QBR_Days_Ago'] > 90).astype(int)
df['flag_tickets']    = (df['Support_Tickets_Open'] >= 3).astype(int)
df['flag_nps']        = (df['NPS_Score'] < 0).astype(int)
df['flag_payment']    = (df['Payment_Overdue_Days'] > 0).astype(int)
df['flag_training']   = (df['Training_Completion_pct'] < 40).astype(int)

FLAG_COLS = [c for c in df.columns if c.startswith('flag_')]
df['Red_Flag_Count'] = df[FLAG_COLS].sum(axis=1)
df['Risk_Tier_B']    = pd.cut(df['Red_Flag_Count'], bins=[-1, 1, 3, len(FLAG_COLS)],
                               labels=['Healthy', 'Watch', 'At Risk'], include_lowest=True)

# Summary
print("=== Customers by risk tier ===")
print(df['Risk_Tier_B'].value_counts().sort_index())
print("\n=== Flag frequency (% of customers) ===")
print((df[FLAG_COLS].mean() * 100).round(1).sort_values(ascending=False))

# First 20 customers: key fields → calculated fields → raw fields
KEY  = ['Customer_ID', 'Company_Name', 'Industry', 'ACV_USD']
CALC = FLAG_COLS + ['Red_Flag_Count', 'Risk_Tier_B']
RAW  = ['Seats_Licensed', 'Seats_Active_30d', 'Logins_Total_30d',
        'Features_Adopted', 'Last_QBR_Days_Ago', 'Support_Tickets_Open',
        'NPS_Score', 'Payment_Overdue_Days', 'Training_Completion_pct']
print("\n=== All customers with calculated columns ===")
display(df[KEY + CALC + RAW].reset_index(drop=True))



---
## Option C - Signal Ranking by Churn Separation

This approach answers a different question: **which signals actually matter?** Instead of assigning weights up front, we let the data tell us which signals best distinguish customers who churned from those who stayed.

---

### Step 1 - Compare averages between churned and healthy customers

For each signal, we split the dataset into two groups - customers who **churned** and customers who **did not** - and calculate the average (mean) value of that signal in each group.

For example, if churned customers had an average seat utilisation of 22% while healthy customers averaged 81%, that is a large gap. Seat utilisation is clearly connected to churn.

---

### Step 2 - Normalise the gap so signals can be compared fairly

The gap between group averages is not enough on its own. A gap of 10 means something very different for a signal measured in days (Last QBR) versus a percentage (Training Completion).

To make signals comparable, we divide the gap by the **standard deviation** - a measure of how spread out the values are across all customers. Think of it as the "typical" amount of variation in that signal.

```
Normalised gap = (mean_churned − mean_not_churned) / standard_deviation
```



This gives a gap measured in "standard units" rather than the original units of the signal. A normalised gap of 1.5 means churned customers sit 1.5 standard deviations away from the average - a strong and meaningful difference.

---

### Step 3 - Rank signals by absolute gap

We sort by the absolute value of the gap (ignoring direction). A signal can indicate churn either by being **too low** (e.g. seat utilisation, logins - negative gap) or **too high** (e.g. open tickets, QBR days - positive gap). Both are equally useful predictors.

---

### Step 4 - Z-scores: where does each customer sit?

For each signal, a **z-score** expresses how far a specific customer's value is from the overall average, measured in standard deviations.

- **z = 0**: exactly average
- **z = +2**: two standard deviations above average (unusually high)
- **z = −2**: two standard deviations below average (unusually low)

For signals like open tickets, a high positive z-score is a warning sign. For signals like seat utilisation, a large negative z-score is the concern. The code combines these into a single `risk_z_score` per customer.

**Added columns:** `seat_util_rate`, `logins_per_seat`, `zscore_*`, `risk_z_score`
**Output file:** `HealthScore_OptionC.xlsx` (two sheets: Customer Data + Signal Ranking)

In [ ]:
#@title Option C - Signal Ranking by Churn Separation
from google.colab import files
pd.options.display.float_format = '{:.2f}'.format

df['seat_util_rate']  = (df['Seats_Active_30d'] / df['Seats_Licensed'].replace(0, 1)).round(1)
df['logins_per_seat'] = (df['Logins_Total_30d']  / df['Seats_Active_30d'].replace(0, 1)).round(1)

SIGNALS = [
    'seat_util_rate', 'logins_per_seat', 'Features_Adopted',
    'Last_QBR_Days_Ago', 'Support_Tickets_Open', 'NPS_Score',
    'Payment_Overdue_Days', 'Training_Completion_pct',
]

# Normalised gap = (mean_churned - mean_not_churned) / overall_std_dev
is_churned = df['Outcome'] == 'Churned'

rows = []
for sig in SIGNALS:
    std = df[sig].std()
    gap = (df.loc[is_churned, sig].mean() - df.loc[~is_churned, sig].mean()) / std if std else 0
    rows.append({
        'Signal':           sig,
        'Mean':             df[sig].mean(),
        'Mean_Churned':     round(df.loc[is_churned,  sig].mean(), 2),
        'Mean_Not_Churned': round(df.loc[~is_churned, sig].mean(), 2),
        'Std_Devlation':    round(std, 2),
        'Normalised_Gap':   round(gap, 2)
    })

rank_df = (pd.DataFrame(rows)
             .assign(Abs_Gap=lambda x: x['Normalised_Gap'].abs())
             .sort_values('Abs_Gap', ascending=False)
             .drop(columns='Abs_Gap')
             .reset_index(drop=True))
rank_df.index += 1

print("=== Signal ranking (strongest churn separator first) ===")
display(rank_df)

# Z-scores: how far each customer sits from the group average (rounded to 1 dp)
for sig in SIGNALS:
    df[f'zscore_{sig}'] = ((df[sig] - df[sig].mean()) / df[sig].std()).round(1)

z_cols = [f'zscore_{s}' for s in SIGNALS]
df['risk_z_score'] = df[z_cols].apply(
    lambda row: sum(-z if s in ['seat_util_rate', 'logins_per_seat', 'Features_Adopted',
                                 'NPS_Score', 'Training_Completion_pct']
                    else z
                    for z, s in zip(row, SIGNALS)), axis=1
).round(1)

# First 20 customers: key fields → calculated fields → raw fields
KEY  = ['Customer_ID', 'Company_Name', 'Industry', 'ACV_USD']
CALC = ['seat_util_rate', 'logins_per_seat'] + z_cols + ['risk_z_score']
RAW  = ['Seats_Licensed', 'Seats_Active_30d', 'Logins_Total_30d',
        'Features_Adopted', 'Last_QBR_Days_Ago', 'Support_Tickets_Open',
        'NPS_Score', 'Payment_Overdue_Days', 'Training_Completion_pct', 'Outcome']
print("\n=== All customers with calculated columns ===")
display(df[KEY + CALC + RAW].reset_index(drop=True))



---
## Which Option Should You Use?

| | Option A - Weighted Score | Option B - Red Flag Counter | Option C - Signal Ranking |
|---|---|---|---|
| **Best for** | CSM dashboards, exec reporting, QBR prep | Daily triage, early alerts, onboarding checks | Calibrating Option A weights, annual strategy review |
| **Data needed** | Current snapshot | Current snapshot | Historical outcomes (Churned / Renewed labels) |
| **Key risk** | Weights are subjective without calibration | All flags carry equal weight | Correlation ≠ causation |
| **Output** | `Health_Score`, `Risk_Tier_A` | `Red_Flag_Count`, `Risk_Tier_B` | `zscore_*`, `risk_z_score` + Signal Ranking sheet |

In practice, teams may use all three together:
- **Option C** to discover which signals matter most,
- **Option A** to operationalise a composite score, and
- **Option B** to give CSMs a simple daily checklist.

---
*Course: Enterprise Architecture for SaaS Business Models © 2026 Peter Datsichin*